# minGPT

A refresher on **minGPT** — Andrej Karpathy's deliberately tiny (~300-line) PyTorch re-implementation of a GPT (decoder-only Transformer) language model. It exists to be *read*: the whole model, training loop, and sampler fit in your head, so it's the canonical artifact for understanding how GPT actually works under the abstractions of `transformers`.

**Domain:** LLM Inference, Training & Optimization  ·  **from study list**  ·  **runnable:** yes

## 1. What & Why

**minGPT is GPT with the magic removed.** The HuggingFace `transformers` `GPT2LMHeadModel` is thousands of lines of generality (every config flag, every backend, every generation strategy). minGPT throws all of that away and keeps only the irreducible core:

- token + positional **embeddings**,
- a stack of identical **Transformer blocks** (causal self-attention + MLP, each with a residual connection and LayerNorm),
- a final linear **head** projecting back to vocabulary logits,
- an AdamW **training loop** and an autoregressive **sampler**.

**The problem it solves:** "I know roughly what a Transformer is, but I want to see *exactly* what GPT computes, with nothing hidden." Because it's so small you can set a breakpoint anywhere, print a tensor shape, and trace a token from input id to output logit.

**Reach for it when:** learning/teaching how GPT works, prototyping an architectural tweak (different attention, new positional scheme), or as a clean base to fork. **Don't reach for it** when you want to *use* a pretrained model in production — that's `transformers` + a real checkpoint. minGPT is a teaching scale model, not a serving stack. (Karpathy's successor [nanoGPT](https://github.com/karpathy/nanoGPT) is the same idea tuned for actually pretraining GPT-2-scale models efficiently.)

## 2. Mental Model

**A GPT is a residual stream that tokens flow up, refining a "what comes next" guess at every layer.**

```
   input ids  [B, T]
        │  tok_emb(id) + pos_emb[t]        # what is this token + where is it
        ▼
   x  [B, T, n_embd]  ──────────────┐  the "residual stream"
        │                            │
        │   ┌─────── Block × n_layer ─┴──────────────┐
        │   │  x = x + Attn(LayerNorm(x))   # tokens look back at earlier tokens
        │   │  x = x + MLP (LayerNorm(x))   # per-token feature processing
        │   └──────────────────────────────────────┘
        ▼
   LayerNorm → head (Linear to vocab)
        ▼
   logits [B, T, vocab]  → softmax → next-token distribution
```

Two ideas make it "GPT" specifically:

1. **Causal mask** — position `t` may attend to positions `≤ t` only. This is what makes it a *language model*: the prediction at every position depends only on the past, so a single forward pass computes the loss for predicting every next token at once.
2. **Residual stream** — each block *adds* to `x` rather than replacing it. Attention and MLP are read/write operations on a shared bus, which is why GPTs train stably even when very deep.

## 3. Key Concepts

- **Decoder-only / autoregressive** — predicts token `t+1` from tokens `1..t`. No encoder, no cross-attention. Trained with plain cross-entropy on the shifted sequence.
- **Causal self-attention** — for each head, `softmax(QKᵀ/√d_k + mask) · V`. The lower-triangular `mask` sets future positions to `-inf` so they vanish after softmax. Multiple **heads** run in parallel on slices of the embedding and are concatenated.
- **Block** — the repeated unit: `x = x + attn(ln1(x)); x = x + mlp(ln2(x))`. Note **pre-norm** (LayerNorm *before* the sublayer) — the modern, stable arrangement.
- **Positional embedding** — a learned vector per position (minGPT/GPT-2 style), added to the token embedding so attention can tell order. `block_size` is the max context length and fixes the size of this table.
- **`block_size` (context window)** — the longest sequence the model can see; the attention mask and position table are sized to it.
- **Weight tying & init** — GPT-2 ties the input embedding and output head weights and uses a small-normal init; minGPT mirrors these so behavior matches the real thing.
- **Sampling** — generation is a loop: forward the current context, take the last position's logits, optionally apply temperature / top-k, sample one token, append, repeat.

## 4. Setup

minGPT's only dependency is **PyTorch** (CPU is fine for the toy scale used here). You can either `pip install` the package from Karpathy's repo, or — as this notebook does — just paste the ~80 lines of model code, since the whole point is that it's small enough to own.

```bash
# Option A: install the package
pip install git+https://github.com/karpathy/minGPT.git

# Option B (this notebook): just need torch
pip install torch --index-url https://download.pytorch.org/whl/cpu
```

In a notebook you'd run `%pip install torch`. The cell below imports torch and fixes the seed so every run is reproducible.

In [1]:
import math
import torch
import torch.nn as nn
from torch.nn import functional as F

torch.manual_seed(1337)  # Karpathy's lucky seed; makes output reproducible
print("torch:", torch.__version__)
print("device:", "cuda" if torch.cuda.is_available() else "cpu (fine for this notebook)")

torch: 2.12.1
device: cpu (fine for this notebook)


## 5. Worked Examples

### Example 1 — The whole model in one cell

Below is a faithful, compact minGPT. Read it top-to-bottom: a config, one attention module, one block, and the `GPT` wrapper that stitches embeddings → blocks → head together and computes the cross-entropy loss when given targets. The `generate` method is the autoregressive sampler. This is genuinely all GPT is.

In [2]:
class GPTConfig:
    """Tiny by default so it trains in seconds on CPU."""
    def __init__(self, vocab_size, block_size, n_layer=3, n_head=3, n_embd=48):
        self.vocab_size = vocab_size      # number of distinct tokens
        self.block_size = block_size      # max context length
        self.n_layer = n_layer            # number of Transformer blocks
        self.n_head = n_head              # attention heads per block
        self.n_embd = n_embd              # residual-stream / embedding width


class CausalSelfAttention(nn.Module):
    def __init__(self, c):
        super().__init__()
        assert c.n_embd % c.n_head == 0
        self.c_attn = nn.Linear(c.n_embd, 3 * c.n_embd)  # q, k, v in one matmul
        self.c_proj = nn.Linear(c.n_embd, c.n_embd)      # output projection
        self.n_head, self.n_embd = c.n_head, c.n_embd
        # lower-triangular mask: position t may only attend to positions <= t
        mask = torch.tril(torch.ones(c.block_size, c.block_size))
        self.register_buffer("mask", mask.view(1, 1, c.block_size, c.block_size))

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        hs = C // self.n_head                                   # head size
        q = q.view(B, T, self.n_head, hs).transpose(1, 2)       # (B, nh, T, hs)
        k = k.view(B, T, self.n_head, hs).transpose(1, 2)
        v = v.view(B, T, self.n_head, hs).transpose(1, 2)
        att = (q @ k.transpose(-2, -1)) / math.sqrt(hs)         # (B, nh, T, T)
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)                            # attention weights
        y = (att @ v).transpose(1, 2).contiguous().view(B, T, C)
        return self.c_proj(y)


class Block(nn.Module):
    """Pre-norm Transformer block: attention then MLP, each on a residual."""
    def __init__(self, c):
        super().__init__()
        self.ln1, self.ln2 = nn.LayerNorm(c.n_embd), nn.LayerNorm(c.n_embd)
        self.attn = CausalSelfAttention(c)
        self.mlp = nn.Sequential(
            nn.Linear(c.n_embd, 4 * c.n_embd), nn.GELU(),
            nn.Linear(4 * c.n_embd, c.n_embd),
        )

    def forward(self, x):
        x = x + self.attn(self.ln1(x))   # tokens exchange information
        x = x + self.mlp(self.ln2(x))    # per-token feature processing
        return x


class GPT(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.block_size = c.block_size
        self.tok_emb = nn.Embedding(c.vocab_size, c.n_embd)
        self.pos_emb = nn.Parameter(torch.zeros(1, c.block_size, c.n_embd))
        self.blocks = nn.ModuleList([Block(c) for _ in range(c.n_layer)])
        self.ln_f = nn.LayerNorm(c.n_embd)
        self.head = nn.Linear(c.n_embd, c.vocab_size, bias=False)
        self.head.weight = self.tok_emb.weight  # weight tying (GPT-2 style)

    def forward(self, idx, targets=None):
        B, T = idx.size()
        assert T <= self.block_size, "sequence longer than block_size"
        x = self.tok_emb(idx) + self.pos_emb[:, :T, :]
        for block in self.blocks:
            x = block(x)
        logits = self.head(self.ln_f(x))                 # (B, T, vocab)
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)),
                                   targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.block_size:]             # crop to context
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature          # last step only
            if top_k is not None:
                v, _ = torch.topk(logits, top_k)
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat([idx, idx_next], dim=1)
            if idx.size(1) > self.block_size and False:
                break
        return idx


# Instantiate a tiny model and run one forward pass on random tokens.
cfg = GPTConfig(vocab_size=10, block_size=16)
model = GPT(cfg)
n_params = sum(p.numel() for p in model.parameters())
x = torch.randint(0, cfg.vocab_size, (2, cfg.block_size))   # batch of 2 sequences
logits, _ = model(x)
print(f"parameters : {n_params:,}")
print(f"input ids  : {tuple(x.shape)}  ->  logits {tuple(logits.shape)}")
print("logits[0, -1] (next-token scores for seq 0):")
print(logits[0, -1].detach().round(decimals=2))

parameters : 86,160
input ids  : (2, 16)  ->  logits (2, 16, 10)
logits[0, -1] (next-token scores for seq 0):
tensor([ 41.5700,  -5.5000, -11.5800,  -1.1500,  -0.9700,  14.3700,  -1.5200,
         -3.9000,   9.3300,  -3.8800])


### Example 2 — Train it on a tiny char-level corpus

Now let's actually *train* the model so the loss means something. We use a character-level vocabulary over a short repeating string. The model should drive the cross-entropy loss far down and learn to continue the pattern — a miniature version of exactly how GPT-2/3 were trained, just with bytes instead of a billion-token web crawl.

In [3]:
# --- a minimal char-level dataset -------------------------------------------
text = "hello minGPT, hello minGPT, the smallest gpt that learns. " * 8
chars = sorted(set(text))
stoi = {ch: i for i, ch in enumerate(chars)}
itos = {i: ch for ch, i in stoi.items()}
data = torch.tensor([stoi[ch] for ch in text], dtype=torch.long)

block_size = 24
cfg = GPTConfig(vocab_size=len(chars), block_size=block_size,
                n_layer=3, n_head=3, n_embd=48)
gpt = GPT(cfg)

def get_batch(batch_size=32):
    ix = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    xb = torch.stack([data[i:i + block_size] for i in ix])
    yb = torch.stack([data[i + 1:i + block_size + 1] for i in ix])  # shifted by 1
    return xb, yb

opt = torch.optim.AdamW(gpt.parameters(), lr=3e-3)
print(f"vocab={len(chars)} chars, {sum(p.numel() for p in gpt.parameters()):,} params\n")

for step in range(301):
    xb, yb = get_batch()
    _, loss = gpt(xb, yb)
    opt.zero_grad(set_to_none=True)
    loss.backward()
    opt.step()
    if step % 75 == 0:
        print(f"step {step:3d}  loss {loss.item():.3f}")
print(f"\nfinal loss {loss.item():.3f}  (random baseline ~ {math.log(len(chars)):.3f})")

vocab=19 chars, 86,976 params

step   0  loss 38.068


step  75  loss 0.331


step 150  loss 0.095


step 225  loss 0.057


step 300  loss 0.153

final loss 0.153  (random baseline ~ 2.944)


In [4]:
# --- sample from the trained model ------------------------------------------
gpt.eval()
context = torch.tensor([[stoi["h"]]], dtype=torch.long)  # prime with 'h'
out = gpt.generate(context, max_new_tokens=60, temperature=0.8, top_k=5)[0].tolist()
print("greedy-ish continuation from 'h':")
print(repr("".join(itos[i] for i in out)))

greedy-ish continuation from 'h':
'helllarns. helo minGPT, helllo minGPT, the smalest gpt theat '


The loss falls from the ~`log(vocab)` random baseline toward zero, and the sample reproduces the training pattern — proof the forward pass, loss, backward pass, and sampler all wire together correctly. Scale `n_layer`/`n_embd`/data up and this same code is nanoGPT pretraining Shakespeare or GPT-2.

## 6. Gotchas & Pitfalls

- **Targets must be inputs shifted by one.** `y[t] = x[t+1]`. Off-by-one here trains the model to copy the current token (loss looks fine, generation is garbage). This is the single most common bug.
- **`block_size` is a hard ceiling.** The learned position table and the causal mask are sized to `block_size`; feeding a longer sequence indexes out of bounds. The sampler must crop context with `idx[:, -block_size:]`.
- **Forgetting the causal mask** turns it into a bidirectional encoder (BERT-like). It will reach near-zero loss on training data by *peeking at the answer*, then generate nonsense — a silent, dangerous failure.
- **Pre-norm vs post-norm.** Modern GPT (and minGPT) put LayerNorm *inside* the residual branch (`x + sublayer(ln(x))`). The original 2017 Transformer used post-norm, which needs learning-rate warmup to train deep stacks stably. Don't mix them up.
- **`model.eval()` / `torch.no_grad()` for sampling.** Not strictly required here (no dropout in this trimmed version), but real minGPT has dropout — sampling in train mode injects noise, and skipping `no_grad` wastes memory building a graph you never backprop.
- **Tiny data overfits instantly.** The toy run above *memorizes* its string — that's the point of a demo, not a sign of a good model. Real training needs far more data than parameters.
- **CPU is fine for toys, not for scale.** This notebook is seconds on CPU; a GPT-2-small pretrain wants a GPU and mixed precision. minGPT is for understanding; use nanoGPT/`transformers` when you actually need throughput.

## 7. When to Use vs Alternatives

| You want to… | Use | Why |
|---|---|---|
| **Understand** how GPT computes, line by line | **minGPT** | ~300 readable lines; nothing hidden. Teaching-first. |
| **Pretrain** a GPT-2-scale model efficiently | **[nanoGPT](https://github.com/karpathy/nanoGPT)** | Same spirit, but with DDP, mixed precision, `torch.compile`, GPT-2 checkpoint loading. minGPT's production-minded sequel. |
| **Use** a pretrained model in an app | **🤗 `transformers`** | Battle-tested, every architecture, generation utilities, ecosystem. Don't hand-roll inference. |
| **Fast inference / serving** at scale | **vLLM / TGI / llama.cpp** | Paged-attention, batching, quantization — none of which minGPT attempts. |
| **Prototype an architecture tweak** | **minGPT (fork it)** | Small enough to fully own and modify; no abstraction to fight. |

**Honest trade-off:** minGPT is intentionally *not* fast, *not* feature-complete, and *not* meant for production. Its value is pedagogical clarity. The moment you care about speed, scale, or robustness, graduate to nanoGPT (training) or `transformers`/vLLM (using). See also the [`flash-attention`](./flash-attention.ipynb), [`kv-cache`](./kv-cache.ipynb), and [`finetune-transformer-lm`](./finetune-transformer-lm.ipynb) notebooks for the optimizations real systems add on top of this core.

## 8. Resources

- **minGPT repo** — the source this notebook distills: <https://github.com/karpathy/minGPT>
- **nanoGPT** — the efficient successor for actual pretraining: <https://github.com/karpathy/nanoGPT>
- **"Let's build GPT from scratch" (Karpathy, video)** — 2-hour line-by-line walkthrough of essentially this code: <https://www.youtube.com/watch?v=kCc8FmEb1nY>
- **"Attention Is All You Need"** — the original Transformer paper: <https://arxiv.org/abs/1706.03762>
- **"Language Models are Unsupervised Multitask Learners" (GPT-2)** — the architecture minGPT mirrors: <https://cdn.openai.com/better-language-models/language_models_are_unsupervised_multitask_learners.pdf>
- **The Illustrated GPT-2 (Jay Alammar)** — visual intuition for the same components: <https://jalammar.github.io/illustrated-gpt2/>